# Project 1 - Milestone 3 (Final Code)
- David Koyrakh
- Bellevue University
- DSC-680: Applied Data Science
- Professor Iranitalab
- Submitted on October 5, 2025


This notebook supports the final white paper by producing defensible figures and model outputs. We begin by:
- Importing core analysis and plotting libraries
- Setting a consistent visual theme for figures
- Loading the provided `heart_disease_uci.csv` into a `pandas` DataFrame for downstream EDA and modeling

In [1]:
# Imports and dataset load
from __future__ import annotations

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Ensure deterministic plots and consistent style
sns.set_theme(context="notebook", style="whitegrid", palette="deep")
np.random.seed(42)

# Use the notebook directory as the project root
NB_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
DATA_PATH = NB_DIR / "data"

csv_path = DATA_PATH / "heart_disease_uci.csv"
assert csv_path.exists(), f"Expected dataset at {csv_path}"

# Load CSV
# Keep strings as objects; do not infer categorical yet; preserve NA
_df = pd.read_csv(csv_path)

# Basic shape sanity check
print({
    "rows": _df.shape[0],
    "cols": _df.shape[1],
    "columns": list(_df.columns),
})

# Quick head/tail for confirmation
display(_df.head(3))
display(_df.tail(3))


{'rows': 920, 'cols': 16, 'columns': ['id', 'age', 'sex', 'dataset', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalch', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'num']}


,id,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,num
0,1,63,Male,Cleveland,typical angina,145.0,233.0,True,lv hypertrophy,150.0,False,2.3,downsloping,0.0,fixed defect,0
1,2,67,Male,Cleveland,asymptomatic,160.0,286.0,False,lv hypertrophy,108.0,True,1.5,flat,3.0,normal,2
2,3,67,Male,Cleveland,asymptomatic,120.0,229.0,False,lv hypertrophy,129.0,True,2.6,flat,2.0,reversable defect,1


,id,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,num
917,918,55,Male,VA Long Beach,asymptomatic,122.0,223.0,True,st-t abnormality,100.0,False,0.0,NaN,NaN,fixed defect,2
918,919,58,Male,VA Long Beach,asymptomatic,NaN,385.0,True,lv hypertrophy,NaN,NaN,NaN,NaN,NaN,NaN,0
919,920,62,Male,VA Long Beach,atypical angina,120.0,254.0,False,lv hypertrophy,93.0,True,0.0,NaN,NaN,NaN,1


# Setup and Data Load - What We Learned

- The CSV loaded successfully from `Semester 5/DSC-680/wk1-4/data/heart_disease_uci.csv`.
- Reported shape matches `previews.txt` expectations: 920 rows × 16 columns.
- Column names align with the preview, including `sex`, `cp`, `restecg`, `thal`, and `num`.
- Mixed types are present (objects for booleans/labels, floats for some numeric fields), which we will normalize next.

Next: audit dtypes and missingness; coerce `fbs`/`exang` to booleans; ensure categorical encodings are consistent with the literature and the Cleveland subset schema.


# Dtypes & Missingness Audit

Before modeling, we need a clear understanding of data types and missing values. This step:
- Profiles dtypes and missingness per column
- Normalizes booleans (`fbs`, `exang`) from string/obj to bool
- Verifies categorical text levels (e.g., `cp`, `restecg`, `thal`, `slope`) to inform encoding

This ensures our later preprocessing (one-hot encoding, scaling) is correct and reproducible.



In [2]:
# Audit and coerce types

def coerce_boolean(series: pd.Series) -> pd.Series:
    truthy = {"true", "t", "yes", "y", "1", 1, True}
    falsy = {"false", "f", "no", "n", "0", 0, False}
    def _map(v):
        if pd.isna(v):
            return np.nan
        if isinstance(v, str):
            lv = v.strip().lower()
            if lv in truthy:
                return True
            if lv in falsy:
                return False
        if v in truthy:
            return True
        if v in falsy:
            return False
        return np.nan
    out = series.map(_map)
    return out.astype("boolean")

# Work on a copy for auditing
_df2 = _df.copy()

# Coerce fbs, exang to booleans
for col in ["fbs", "exang"]:
    if col in _df2.columns:
        _df2[col] = coerce_boolean(_df2[col])

# Basic dtype report
dtype_report = _df2.dtypes.astype(str).to_frame("dtype")
missing_report = _df2.isna().sum().to_frame("num_missing")
share_missing = (_df2.isna().mean() * 100.0).round(2).to_frame("pct_missing")
profile = dtype_report.join(missing_report).join(share_missing).sort_values(["pct_missing", "num_missing"], ascending=False)

print("Schema profile (top 10 by pct missing):")
display(profile.head(10))

# Preview categorical levels for selected columns
categorical_cols = ["sex", "dataset", "cp", "restecg", "slope", "thal"]
levels = {c: sorted(_df2[c].dropna().astype(str).unique().tolist()) for c in categorical_cols if c in _df2.columns}
print("Categorical levels (preview):")
for c, vals in levels.items():
    print(f"- {c}: {vals[:12]}{' ...' if len(vals) > 12 else ''}")

# Persist the working frame for subsequent steps
_df = _df2


Schema profile (top 10 by pct missing):


,dtype,num_missing,pct_missing
ca,float64,611,66.41
thal,object,486,52.83
slope,object,309,33.59
fbs,boolean,90,9.78
oldpeak,float64,62,6.74
trestbps,float64,59,6.41
thalch,float64,55,5.98
exang,boolean,55,5.98
chol,float64,30,3.26
restecg,object,2,0.22


Categorical levels (preview):
- sex: ['Female', 'Male']
- dataset: ['Cleveland', 'Hungary', 'Switzerland', 'VA Long Beach']
- cp: ['asymptomatic', 'atypical angina', 'non-anginal', 'typical angina']
- restecg: ['lv hypertrophy', 'normal', 'st-t abnormality']
- slope: ['downsloping', 'flat', 'upsloping']
- thal: ['fixed defect', 'normal', 'reversable defect']


# Dtypes & Missingness - What We Learned

- `fbs` and `exang` successfully coerced to proper boolean dtype.
- Several categorical columns are text labels (`cp`, `restecg`, `slope`, `thal`). We'll one‑hot encode later.
- Highest missingness is concentrated in non‑Cleveland rows and in fields like `ca`, `oldpeak`, and some categorical diagnostics for external cohorts.

Next: focus the modeling dataset on the Cleveland cohort and derive a binary target (`num > 0`).


# Cleveland Subset & Binary Target

Historically, most published benchmarking uses the Cleveland subset with a binary target. We will:
- Filter `dataset == 'Cleveland'`
- Create `target = (num > 0).astype(int)` where 1 indicates presence of heart disease
- Summarize class balance and basic descriptive stats to ensure suitability for modeling



In [3]:
# Cleveland subset and binary target
import json

NB_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
OUT_PATH = NB_DIR / "outputs"
OUT_PATH.mkdir(parents=True, exist_ok=True)

cleveland = _df[_df["dataset"].astype(str) == "Cleveland"].copy()
cleveland["target"] = (cleveland["num"] > 0).astype(int)

n_rows, n_cols = cleveland.shape
class_counts = cleveland["target"].value_counts().sort_index()
class_share = (class_counts / n_rows).round(4)

print({
    "cleveland_rows": int(n_rows),
    "cleveland_cols": int(n_cols),
    "class_counts": class_counts.to_dict(),
    "class_share": class_share.to_dict(),
})

display(pd.DataFrame({
    "count": class_counts,
    "share": class_share
}).rename(index={0: "no_disease", 1: "disease"}))

# Save artifacts for downstream steps
with open(OUT_PATH / "cleveland_summary.json", "w") as f:
    json.dump({
        "n_rows": int(n_rows),
        "n_cols": int(n_cols),
        "class_counts": {str(int(k)): int(v) for k, v in class_counts.items()},
        "class_share": {str(int(k)): float(v) for k, v in class_share.items()},
    }, f, indent=2)

cleveland.to_pickle(OUT_PATH / "cleveland.pkl")

# Keep for later stages
_cleveland = cleveland


{'cleveland_rows': 304, 'cleveland_cols': 17, 'class_counts': {0: 165, 1: 139}, 'class_share': {0: 0.5428, 1: 0.4572}}


,count,share
target,,
no_disease,165,0.5428
disease,139,0.4572


# Cleveland - What We Learned

- Focused dataset size: 304 rows × 17 columns (Cleveland only).
- Class balance: 165 no-disease (54.28%), 139 disease (45.72%).
- The binary target `target` aligns with common literature practice (`num > 0`).

Next: targeted EDA visuals for the white paper (class balance bar, age distribution by target, chest pain type by target).

# EDA Visuals

We will produce three white‑paper‑ready visuals:
- Class balance bar plot for `target`
- Age distribution by `target`
- Chest pain type (`cp`) by `target`

Figures will be saved under `outputs/`.

In [4]:
# Generate EDA figures and save to outputs
from pathlib import Path

NB_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
OUT = NB_DIR / "outputs"
OUT.mkdir(parents=True, exist_ok=True)

# 1) Class balance
fig1, ax1 = plt.subplots(figsize=(5, 4))
class_counts = _cleveland["target"].value_counts().sort_index()
ax1.bar(["no_disease", "disease"], [class_counts.get(0, 0), class_counts.get(1, 0)], color=["#4C78A8", "#F58518"])
ax1.set_title("Heart Disease - Class Balance (Cleveland)")
ax1.set_ylabel("Count")
fig1.tight_layout()
fig1_path = OUT / "fig_class_balance.png"
fig1.savefig(fig1_path, dpi=200)
plt.close(fig1)

# 2) Age distribution by target
fig2, ax2 = plt.subplots(figsize=(6, 4))
sns.kdeplot(data=_cleveland, x="age", hue="target", common_norm=False, fill=True, palette=["#4C78A8", "#F58518"], ax=ax2)
ax2.set_title("Age Distribution by Target (Cleveland)")
fig2.tight_layout()
fig2_path = OUT / "fig_age_by_target.png"
fig2.savefig(fig2_path, dpi=200)
plt.close(fig2)

# 3) Chest pain type by target
fig3, ax3 = plt.subplots(figsize=(7, 4))
cp_order = sorted(_cleveland["cp"].dropna().unique().tolist())
sns.countplot(data=_cleveland, y="cp", hue="target", order=cp_order, palette=["#4C78A8", "#F58518"], ax=ax3)
ax3.set_title("Chest Pain Type by Target (Cleveland)")
ax3.set_xlabel("Count")
fig3.tight_layout()
fig3_path = OUT / "fig_cp_by_target.png"
fig3.savefig(fig3_path, dpi=200)
plt.close(fig3)

print({"saved": [str(fig1_path), str(fig2_path), str(fig3_path)]})


{'saved': ['BU\\Semester 5\\DSC-680\\wk1-4\\outputs\\fig_class_balance.png', 'BU\\Semester 5\\DSC-680\\wk1-4\\outputs\\fig_age_by_target.png', 'BU\\Semester 5\\DSC-680\\wk1-4\\outputs\\fig_cp_by_target.png']}


# EDA - What We Learned

- Class balance is moderately imbalanced toward no-disease (≈54% vs 46%).
- Age distributions by target overlap but suggest older ages trend toward disease.
- Chest pain type `asymptomatic` is more prevalent among disease cases; `typical angina` skews non-disease.

We will proceed to preprocessing and a transparent baseline (logistic regression).

# Preprocessing + Logistic Regression

We build a transparent baseline using a `scikit-learn` pipeline:
- Split Cleveland data (stratified) into train/test
- One-hot encode categorical features; scale numeric
- Exclude leakage-prone columns (`id`, `dataset`, and the multi-class `num`)
- Train `LogisticRegression` with class_weight="balanced"
- Persist metrics and confusion matrix/ROC for the white paper



In [5]:
# Build and evaluate logistic regression baseline
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, RocCurveDisplay
)
import json

NB_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
OUT = NB_DIR / "outputs"
OUT.mkdir(parents=True, exist_ok=True)

# Features/target
df = _cleveland.copy()
# Exclude leakage-prone columns: 'num' (multiclass label feeding target), 'id', 'dataset'
X = df.drop(columns=["target", "num", "id", "dataset"], errors="ignore")
y = df["target"].astype(int)

# Identify feature types
categorical_cols = [c for c in X.columns if X[c].dtype == "object"]
boolean_cols = [c for c in X.columns if str(X[c].dtype) == "boolean"]
numeric_cols = [c for c in X.columns if c not in categorical_cols + boolean_cols]

# Simple imputation: numeric->median, categorical/bool->most_frequent
from sklearn.impute import SimpleImputer

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler())
        ]), numeric_cols),
        ("cat", Pipeline(steps=[
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), categorical_cols + boolean_cols),
    ], remainder="drop"
)

clf = Pipeline(steps=[
    ("prep", preprocessor),
    ("model", LogisticRegression(max_iter=200, class_weight="balanced", solver="liblinear"))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

clf.fit(X_train, y_train)

# Predictions and metrics
y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]

metrics = {
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "precision": float(precision_score(y_test, y_pred)),
    "recall": float(recall_score(y_test, y_pred)),
    "f1": float(f1_score(y_test, y_pred)),
    "roc_auc": float(roc_auc_score(y_test, y_prob)),
}

cm = confusion_matrix(y_test, y_pred).tolist()

# Save metrics and plots
with open(OUT / "logreg_metrics.json", "w") as f:
    json.dump({"metrics": metrics, "confusion_matrix": cm}, f, indent=2)

fig, ax = plt.subplots(figsize=(5, 4))
RocCurveDisplay.from_predictions(y_test, y_prob, ax=ax)
ax.set_title("Logistic Regression ROC (Cleveland)")
fig.tight_layout()
fig.savefig(OUT / "logreg_roc.png", dpi=200)
plt.close(fig)

print(metrics)
print({"confusion_matrix": cm})


{'accuracy': 0.868421052631579, 'precision': 0.8378378378378378, 'recall': 0.8857142857142857, 'f1': 0.8611111111111112, 'roc_auc': 0.9358885017421603}
{'confusion_matrix': [[35, 6], [4, 31]]}


# Logistic Regression

- Accuracy 0.868, Precision 0.838, Recall 0.886, F1 0.861, ROC‑AUC 0.936.
- Confusion matrix: TN=35, FP=6, FN=4, TP=31.
- Leakage mitigated by dropping `id`, `dataset`, and `num`.

In [6]:
# Train and evaluate RandomForest baseline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import RocCurveDisplay
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
import json

NB_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
OUT = NB_DIR / "outputs"

# Reuse the same split logic
df = _cleveland.copy()
# Exclude leakage-prone columns: 'num', 'id', 'dataset'
X = df.drop(columns=["target", "num", "id", "dataset"], errors="ignore")
y = df["target"].astype(int)

categorical_cols = [c for c in X.columns if X[c].dtype == "object"]
boolean_cols = [c for c in X.columns if str(X[c].dtype) == "boolean"]
numeric_cols = [c for c in X.columns if c not in categorical_cols + boolean_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler())
        ]), numeric_cols),
        ("cat", Pipeline(steps=[
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), categorical_cols + boolean_cols),
    ], remainder="drop"
)

rf = Pipeline(steps=[
    ("prep", preprocessor),
    ("model", RandomForestClassifier(n_estimators=300, max_depth=4, random_state=42, class_weight="balanced"))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

metrics = {
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "precision": float(precision_score(y_test, y_pred)),
    "recall": float(recall_score(y_test, y_pred)),
    "f1": float(f1_score(y_test, y_pred)),
    "roc_auc": float(roc_auc_score(y_test, y_prob)),
}

cm = confusion_matrix(y_test, y_pred).tolist()

with open(OUT / "rf_metrics.json", "w") as f:
    json.dump({"metrics": metrics, "confusion_matrix": cm}, f, indent=2)

fig, ax = plt.subplots(figsize=(5, 4))
RocCurveDisplay.from_predictions(y_test, y_prob, ax=ax)
ax.set_title("RandomForest ROC (Cleveland)")
fig.tight_layout()
fig.savefig(OUT / "rf_roc.png", dpi=200)
plt.close(fig)

print(metrics)
print({"confusion_matrix": cm})


{'accuracy': 0.8947368421052632, 'precision': 0.8857142857142857, 'recall': 0.8857142857142857, 'f1': 0.8857142857142857, 'roc_auc': 0.9519163763066202}
{'confusion_matrix': [[37, 4], [4, 31]]}


# RandomForest - Cross-Validation

To obtain a robust generalization estimate and guard against split variance and subtle leakage, we run 5-fold stratified cross-validation on the leakage-mitigated pipeline and report mean and standard deviation across folds.



In [7]:
# RF 5-fold CV
from sklearn.model_selection import StratifiedKFold, cross_validate

NB_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
OUT = NB_DIR / "outputs"

# Build the same RF pipeline with leakage-mitigated features
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

_df = _cleveland.copy()
X = _df.drop(columns=["target", "num", "id", "dataset"], errors="ignore")
y = _df["target"].astype(int)

categorical_cols = [c for c in X.columns if X[c].dtype == "object"]
boolean_cols = [c for c in X.columns if str(X[c].dtype) == "boolean"]
numeric_cols = [c for c in X.columns if c not in categorical_cols + boolean_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler())
        ]), numeric_cols),
        ("cat", Pipeline(steps=[
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), categorical_cols + boolean_cols),
    ], remainder="drop"
)

rf = Pipeline(steps=[
    ("prep", preprocessor),
    ("model", RandomForestClassifier(n_estimators=300, max_depth=4, random_state=42, class_weight="balanced"))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]
res = cross_validate(rf, X, y, cv=cv, scoring=scoring, return_train_score=False)

cv_summary = {m: {"mean": float(res[f"test_{m}"].mean()), "std": float(res[f"test_{m}"].std())} for m in scoring}
with open(OUT / "rf_cv_metrics.json", "w") as f:
    json.dump(cv_summary, f, indent=2)

print(cv_summary)


{'accuracy': {'mean': 0.8289617486338798, 'std': 0.02652384939783955}, 'precision': {'mean': 0.8325902921555096, 'std': 0.02635394577936183}, 'recall': {'mean': 0.7838624338624338, 'std': 0.051932773482293665}, 'f1': {'mean': 0.8066576819407008, 'std': 0.033326919898464945}, 'roc_auc': {'mean': 0.9150953984287318, 'std': 0.02030360202829069}}


# RandomForest - What We Learned

- 5-fold CV (mean ± sd):
  - Accuracy 0.829 ± 0.027, Precision 0.833 ± 0.026, Recall 0.784 ± 0.052, F1 0.807 ± 0.033, ROC‑AUC 0.915 ± 0.020.
- CV results are plausible and below the perfect holdout, suggesting the earlier perfection was a split artifact; we’ll cite CV as the primary generalization estimate.

# Interpretability

We will compute:
- Logistic regression odds ratios (top positive/negative predictors)
- RandomForest feature importances

We’ll save tabular summaries and simple bar plots to `outputs/` for the white paper.

In [8]:
# Compute odds ratios and RF permutation importances; save artifacts
import numpy as np
import pandas as pd

from pathlib import Path

NB_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
OUT = NB_DIR / "outputs"
OUT.mkdir(parents=True, exist_ok=True)

# Refit a fresh logistic pipeline to extract coefficients with the finalized preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

_df = _cleveland.copy()
X = _df.drop(columns=["target", "num", "id", "dataset"], errors="ignore")
y = _df["target"].astype(int)

categorical_cols = [c for c in X.columns if X[c].dtype == "object"]
boolean_cols = [c for c in X.columns if str(X[c].dtype) == "boolean"]
numeric_cols = [c for c in X.columns if c not in categorical_cols + boolean_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler())
        ]), numeric_cols),
        ("cat", Pipeline(steps=[
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), categorical_cols + boolean_cols),
    ], remainder="drop"
)

# Logistic regression: coefficients -> odds ratios
logit = Pipeline(steps=[
    ("prep", preprocessor),
    ("model", LogisticRegression(max_iter=300, class_weight="balanced", solver="liblinear"))
])
logit.fit(X, y)

# Recover feature names after preprocessing for logit
oh = logit.named_steps["prep"].named_transformers_["cat"].named_steps["onehot"]
cat_feature_names = oh.get_feature_names_out(categorical_cols + boolean_cols).tolist()
feature_names = numeric_cols + cat_feature_names
coefs = logit.named_steps["model"].coef_[0]
odds = np.exp(coefs)

coef_df = pd.DataFrame({"feature": feature_names, "coef": coefs, "odds_ratio": odds}).sort_values("odds_ratio", ascending=False)
coef_df.to_csv(OUT / "logreg_odds_ratios.csv", index=False)

# Bar plot of top/bottom predictors
import seaborn as sns
import matplotlib.pyplot as plt

top_k = 10
plot_df = pd.concat([
    coef_df.head(top_k).assign(direction="positive"),
    coef_df.tail(top_k).assign(direction="negative")
])
fig, ax = plt.subplots(figsize=(8, 6))
sns.barplot(data=plot_df, x="odds_ratio", y="feature", hue="direction", dodge=False, palette=["#F58518", "#4C78A8"], ax=ax)
ax.set_title("Logistic Regression - Top/Bottom Odds Ratios")
fig.tight_layout()
fig.savefig(OUT / "logreg_top_bottom_odds.png", dpi=200)
plt.close(fig)

# RandomForest permutation importances aggregated across 5-fold CV
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.inspection import permutation_importance

rf_pipe = Pipeline(steps=[
    ("prep", preprocessor),
    ("model", RandomForestClassifier(n_estimators=300, max_depth=4, random_state=42, class_weight="balanced"))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
feature_to_values: dict[str, list[float]] = {}

for train_idx, valid_idx in cv.split(X, y):
    X_tr, X_va = X.iloc[train_idx], X.iloc[valid_idx]
    y_tr, y_va = y.iloc[train_idx], y.iloc[valid_idx]

    rf_pipe.fit(X_tr, y_tr)

    # Feature names for this fold
    oh_fold = rf_pipe.named_steps["prep"].named_transformers_["cat"].named_steps["onehot"]
    cat_names_fold = oh_fold.get_feature_names_out(categorical_cols + boolean_cols).tolist()
    names_fold = numeric_cols + cat_names_fold

    # Permutation importance on validation fold
    res = permutation_importance(rf_pipe, X_va, y_va, scoring="roc_auc", n_repeats=10, random_state=42)
    importances_fold = res.importances_mean

    # Accumulate
    for fname, val in zip(names_fold, importances_fold):
        feature_to_values.setdefault(fname, []).append(float(val))

# Aggregate
rows = []
for fname, vals in feature_to_values.items():
    rows.append({
        "feature": fname,
        "importance_mean": float(np.mean(vals)),
        "importance_std": float(np.std(vals, ddof=0)),
        "n_folds": int(len(vals)),
    })
perm_df = pd.DataFrame(rows).sort_values("importance_mean", ascending=False)
perm_df.to_csv(OUT / "rf_permutation_importance.csv", index=False)

# Plot top-k permutation importances
fig, ax = plt.subplots(figsize=(8, 6))
sns.barplot(data=perm_df.head(15), x="importance_mean", y="feature", color="#4C78A8", ax=ax)
ax.set_title("RandomForest - Top Permutation Importances (5-fold CV)")
fig.tight_layout()
fig.savefig(OUT / "rf_top_perm_importance.png", dpi=200)
plt.close(fig)


# Interpretability - What We Learned

- Logistic (odds ratios): strongest positive associations include `ca` (~2.99×), `cp_asymptomatic` (~2.92×), `thal_reversable defect` (~2.09×), and `sex_Male` (~1.83×). Protective signals include `sex_Female` (~0.48×), `cp_typical angina` (~0.52×), and higher `thalch` (~0.68×).
- RandomForest importances: top features are `thal_normal`, `ca`, `cp_asymptomatic`, `oldpeak`, `thalch`, and `thal_reversable defect`.

These align with domain expectations: ST/thal abnormalities, vessel count (`ca`), exercise-induced angina/oldpeak, and chest pain presentation are influential.

# Figure Artifacts

Saved under `outputs/`:
- fig_class_balance.png
- fig_age_by_target.png
- fig_cp_by_target.png
- logreg_roc.png
- rf_roc.png
- logreg_top_bottom_odds.png
- rf_top_perm_importance.png


#### What we learned

- Both models perform strongly; RF slightly higher AUC on holdout (≈0.952 vs 0.936).
- Cross-validated AUC for RF ≈ 0.915 suggests robust generalization for this dataset size.
- Interpretability: logistic odds and RF permutation importances agree on risk drivers (cp, ca, thal, ECG markers).
- Next steps before any operational use: calibration, subgroup audits, threshold selection via decision curves, model card.
